# Metadata

Domain: Mathematics

Subdomain: Numerical Integration

References: Adaptive Simpson quadrature and composite Newton-Cotes rules, as covered in Burden & Faires, *Numerical Analysis*; Press et al., *Numerical Recipes*; and Richardson extrapolation as covered in Atkinson, *An Introduction to Numerical Analysis*, Ch. 5.

Dependencies: numpy, math, typing (Callable, Tuple). Subproblem 2 (`richardson_*`) depends on Subproblem 1's `composite_simpson`. The Main Problem (`main_problem`) depends on both `composite_simpson` and the `richardson_error_estimate` family from Subproblem 2 — the general, `r`-parameterized version must be reused, not reimplemented, at the n=2/r=2 special case used at each recursive leaf.

Taxonomy: numerical-integration, composite-quadrature, adaptive-algorithms, richardson-extrapolation, error-estimation, recursion.


# Title

Adaptive Numerical Integration with Simpson's Rule


# Subproblem 1

## Prompt

Implement the composite Simpson's rule for approximating ∫_a^b f(x) dx
using n equal subintervals, for **any** integer n >= 2 (not just even n).

def composite_simpson(f: Callable, a: float, b: float, n: int) -> float:
    """
    Parameters
    ----------
    f : callable
        f(x) -> scalar.
    a, b : float
        Integration interval, a < b.
    n : int
        Number of subintervals. Must be an integer >= 2, even or odd.

    Returns
    -------
    S : float
        Composite Simpson's rule approximation of the integral of f over
        [a, b], using exactly n equal-width subintervals, i.e. evaluating
        f at exactly the n+1 equally spaced nodes a + i*(b-a)/n for
        i = 0, ..., n.

    Raises
    ------
    ValueError
        If a >= b, or if n < 2.
    """


def simpson_weights(a: float, b: float, n: int) -> np.ndarray:
    """
    Parameters
    ----------
    a, b : float
        Integration interval, a < b.
    n : int
        Number of subintervals. Integer >= 2, even or odd.

    Returns
    -------
    weights : np.ndarray, shape (n+1,)
        Raw integration weights such that for any integrand f,
            composite_simpson(f, a, b, n) == sum(weights[i] * f(x_i))
        for the equally spaced nodes x_i = a + i*(b-a)/n. Each weight
        already includes the panel-width factor h = (b-a)/n.

        For even n the vector is (h/3) * [1, 4, 2, 4, ..., 4, 1].

        For odd n, the first n-3 panels contribute 1/3-rule weights and
        the last 3 panels contribute 3/8-rule weights. For odd n >= 5 the
        1/3 head has at least one panel and shares its right endpoint with
        the left endpoint of the 3/8 tail, so the shared transition node
        x_{n-3} carries the combined weight
            h * (1/3 + 3/8) = (17/24) * h.
        For n = 3 the 1/3 head is empty (n-3 = 0 panels): the rule is the
        pure 3/8 rule on [x_0, x_1, x_2, x_3] with weights (3h/8)[1,3,3,1],
        so x_0 = x_{n-3} carries weight 3h/8 (there is no 1/3 contribution
        to combine with). The 17h/24 combined weight applies only when the
        1/3 head is non-empty, i.e. for odd n >= 5.

    Raises
    ------
    ValueError
        If a >= b, or if n < 2.
    """
    """

## Background

When $n$ is even, use the standard composite 1/3 rule on nodes $x_i = a + i h$, $h=(b-a)/n$:

$$S = \frac{h}{3}\left(f(x_0) + f(x_n) + 4\sum_{i \text{ odd}} f(x_i) + 2\sum_{i \text{ even}, 0<i<n} f(x_i)\right).$$

When $n$ is odd, the 1/3 rule alone cannot be applied to the whole interval (it needs an even number of subintervals). Instead, apply the composite 1/3 rule to the first $n-3$ subintervals (always even, since $n$ is odd), and apply Simpson's 3/8 rule to the *last three* subintervals $[x_{n-3}, x_{n-2}, x_{n-1}, x_n]$:

$$S_{3/8} = \frac{3h}{8}\big(f(x_{n-3}) + 3f(x_{n-2}) + 3f(x_{n-1}) + f(x_n)\big),$$

and sum the two contributions. In both the even and odd cases, the node spacing $h=(b-a)/n$ must be computed from the *original* $n$ requested — the full set of $n+1$ nodes $\{a+ih\}_{i=0}^{n}$ must all be evaluated exactly once, since a caller requesting a specific $n$ is relying on that specific resolution.


## Testing Template

In [ ]:
# ============ TEST CASES ============
import numpy as np
import math

def test_case_1():
    import time
    n = 100003
    count = [0]
    def f(x):
        count[0] += 1
        return math.sin(3 * x) + 0.5 * x**2
    t0 = time.time()
    S = composite_simpson(f, 0.0, 1.0, n)
    elapsed = time.time() - t0
    assert count[0] == n + 1, f"expected {n+1} calls, got {count[0]}"
    assert elapsed < 5.0, f"too slow for n={n}: {elapsed:.2f}s"
    h = 1.0 / n
    xs = np.arange(n + 1) * h
    vals = np.sin(3 * xs) + 0.5 * xs**2
    m = n - 3
    w = np.zeros(m + 1); w[0] = w[m] = 1.0; w[1:m:2] = 4.0; w[2:m:2] = 2.0
    oracle = float(np.sum(w * vals[:m+1]) * h / 3.0) + (3*h/8.0)*(vals[m]+3*vals[m+1]+3*vals[m+2]+vals[n])
    assert abs(S - oracle) < 1e-8, f"got {S}, expected {oracle}"
    print('test_case_1 passed')


def test_case_2():
    n = 9999
    S = composite_simpson(lambda x: x**5, 0.0, 1.0, n)
    h = 1.0 / n
    xs = np.arange(n + 1) * h
    vals = xs**5
    m = n - 3
    w = np.zeros(m + 1); w[0] = w[m] = 1.0; w[1:m:2] = 4.0; w[2:m:2] = 2.0
    oracle = float(np.sum(w * vals[:m+1]) * h / 3.0) + (3*h/8.0)*(vals[m]+3*vals[m+1]+3*vals[m+2]+vals[n])
    assert abs(S - oracle) < 5e-11, f"got {S}, expected {oracle}"
    print('test_case_2 passed')


def test_case_3():
    from fractions import Fraction

    def exact_mixed(coeffs, a, b, n):
        a, b = Fraction(a), Fraction(b)
        h = (b - a) / n
        xs = [a + i * h for i in range(n + 1)]
        def f(x):
            val, p = Fraction(0), Fraction(1)
            for c in coeffs:
                val += c * p; p *= x
            return val
        m = n - 3
        total = Fraction(0)
        if m >= 2:
            head = f(xs[0]) + f(xs[m])
            for i in range(1, m):
                head += (4 if i % 2 else 2) * f(xs[i])
            total += h / 3 * head
        tail = f(xs[m]) + 3*f(xs[m+1]) + 3*f(xs[m+2]) + f(xs[n])
        total += 3 * h / 8 * tail
        return total

    n = 31
    exact = float(exact_mixed([0, 0, 0, 0, 0, 1], 0, 1, n))
    S = composite_simpson(lambda x: x**5, 0.0, 1.0, n)
    assert abs(S - exact) < 5e-11, f"n={n} x^5 got {S}, expected {exact}"
    print('test_case_3 passed')


def test_case_4():
    from fractions import Fraction

    def exact_mixed(coeffs, a, b, n):
        a, b = Fraction(a), Fraction(b)
        h = (b - a) / n
        xs = [a + i * h for i in range(n + 1)]
        def f(x):
            val, p = Fraction(0), Fraction(1)
            for c in coeffs:
                val += c * p; p *= x
            return val
        m = n - 3
        total = Fraction(0)
        if m >= 2:
            head = f(xs[0]) + f(xs[m])
            for i in range(1, m):
                head += (4 if i % 2 else 2) * f(xs[i])
            total += h / 3 * head
        tail = f(xs[m]) + 3*f(xs[m+1]) + 3*f(xs[m+2]) + f(xs[n])
        total += 3 * h / 8 * tail
        return total

    n = 9
    exact = float(exact_mixed([0, 0, 0, 0, 0, 1], 0, 1, n))
    S = composite_simpson(lambda x: x**5, 0.0, 1.0, n)
    assert abs(S - exact) < 5e-11, f"n={n} x^5 got {S}, expected {exact}"
    print('test_case_4 passed')


def test_case_5():
    a, b, n = 0.0, 1.0, 5
    h = (b - a) / n
    w = simpson_weights(a, b, n)
    m = n - 3
    assert np.isclose(w[m], 17*h/24, atol=1e-13), f"w[{m}]={w[m]}, expected 17h/24={17*h/24}"
    assert not np.isclose(w[m], h/3, atol=1e-6)
    assert not np.isclose(w[m], 3*h/8, atol=1e-6)
    print('test_case_5 passed')


def test_case_6():
    from fractions import Fraction

    def exact_mixed(coeffs, a, b, n):
        a, b = Fraction(a), Fraction(b)
        h = (b - a) / n
        xs = [a + i * h for i in range(n + 1)]
        def f(x):
            val, p = Fraction(0), Fraction(1)
            for c in coeffs:
                val += c * p; p *= x
            return val
        m = n - 3
        total = Fraction(0)
        if m >= 2:
            head = f(xs[0]) + f(xs[m])
            for i in range(1, m):
                head += (4 if i % 2 else 2) * f(xs[i])
            total += h / 3 * head
        tail = f(xs[m]) + 3*f(xs[m+1]) + 3*f(xs[m+2]) + f(xs[n])
        total += 3 * h / 8 * tail
        return total

    a, b, n = -3.0, 2.0, 9
    exact = float(exact_mixed([0, 0, 0, 0, 0, 1], a, b, n))
    S = composite_simpson(lambda x: x**5, a, b, n)
    assert abs(S - exact) < 5e-11, f"n={n} x^5 on [{a},{b}] got {S}, expected {exact}"
    print('test_case_6 passed')


def test_case_7():
    a, b, n = 0.0, 1.0, 9973
    h = (b - a) / n
    m = n - 3
    w = simpson_weights(a, b, n)
    assert len(w) == n + 1
    assert np.isclose(w.sum(), b - a, atol=1e-9), f"weight sum {w.sum()} != {b-a}"
    assert np.isclose(w[m], 17 * h / 24, atol=1e-13), f"transition weight wrong at m={m}"
    f = lambda x: np.sin(x) + 0.3 * x**2
    xs = np.linspace(a, b, n + 1)
    assert np.isclose(float(np.dot(w, f(xs))), composite_simpson(f, a, b, n), atol=1e-9)
    print('test_case_7 passed')


def test_case_9():
    f = lambda x: math.exp(-x) * math.cos(5 * x)
    F = lambda x: math.exp(-x) * (5 * math.sin(5 * x) - math.cos(5 * x)) / 26.0
    a, b = 0.0, 3.0
    exact = F(b) - F(a)
    S25 = composite_simpson(f, a, b, 25)
    S50 = composite_simpson(f, a, b, 50)
    err25 = abs(S25 - exact)
    err50 = abs(S50 - exact)
    assert err25 < 5e-4, f"n=25 err={err25:.3e} too large"
    assert err50 < 5e-4, f"n=50 err={err50:.3e} too large"
    assert err50 < err25
    print('test_case_9 passed')


def test_case_10():
    a, b, n = 0.0, 1e-9, 101
    S = composite_simpson(lambda x: x**3, a, b, n)
    exact = (b**4 - a**4) / 4.0
    assert abs(S - exact) < 1e-9 * abs(exact) + 1e-40, f"got {S}, expected {exact}"
    w = simpson_weights(a, b, n)
    assert np.isclose(w.sum(), b - a, rtol=1e-9)
    print('test_case_10 passed')


def test_case_11():
    f = lambda x: 1e12 * x**3 - 1e-8 * x
    S = composite_simpson(f, 0.0, 1.0, 4)
    exact = 1e12 / 4.0 - 1e-8 / 2.0
    assert abs(S - exact) / abs(exact) < 1e-9, f"got {S}, expected {exact}"
    print('test_case_11 passed')


def test_case_12():
    n = 50021
    a, b = -3.7, 2.9
    S = composite_simpson(lambda x: x**5, a, b, n)
    h = (b - a) / n
    xs = a + np.arange(n + 1) * h
    vals = xs**5
    m = n - 3
    w = np.zeros(m + 1); w[0] = w[m] = 1.0; w[1:m:2] = 4.0; w[2:m:2] = 2.0
    oracle = float(np.sum(w * vals[:m+1]) * h / 3.0) + (3*h/8.0)*(vals[m]+3*vals[m+1]+3*vals[m+2]+vals[n])
    assert abs(S - oracle) < 1e-9, f"got {S}, expected {oracle}"
    print('test_case_12 passed')


def test_case_13():
    def strict_f(x):
        if type(x) is not float:
            raise TypeError(f"expected python float, got {type(x).__name__}")
        return x**2
    S = composite_simpson(strict_f, 0.0, 1.0, 7)
    assert np.isfinite(S)
    print('test_case_13 passed')


def test_case_14():
    n = 50001
    a, b = 0.0, 1.0
    h = (b - a) / n
    seen = []
    def f(x):
        seen.append(x)
        return x
    composite_simpson(f, a, b, n)
    assert len(seen) == n + 1, f"expected {n+1} calls, got {len(seen)}"
    expected_nodes = sorted(a + i * h for i in range(n + 1))
    assert np.allclose(sorted(seen), expected_nodes, rtol=0, atol=1e-9)
    print('test_case_14 passed')


def test_case_15():
    n = 27
    a, b = 0.0, 2.3
    S = composite_simpson(lambda x: math.sin(37 * x * x), a, b, n)
    h = (b - a) / n
    xs = a + np.arange(n + 1) * h
    vals = np.sin(37 * xs * xs)
    m = n - 3
    w = np.zeros(m + 1); w[0] = w[m] = 1.0; w[1:m:2] = 4.0; w[2:m:2] = 2.0
    oracle = float(np.sum(w * vals[:m+1]) * h / 3.0) + (3*h/8.0)*(vals[m]+3*vals[m+1]+3*vals[m+2]+vals[n])
    assert abs(S - oracle) < 1e-9, f"got {S}, expected {oracle}"
    print('test_case_15 passed')


def test_case_16():
    # n = 3 boundary: the 1/3 head is empty (n-3 = 0 panels), so the mixed rule is the PURE
    # 3/8 rule with weights (3h/8)[1, 3, 3, 1]. In particular x_0 = x_{n-3} carries weight
    # 3h/8, NOT the 17h/24 combined weight (which only applies for odd n >= 5, where the 1/3
    # head shares a node with the 3/8 tail).
    a, b, n = 0.0, 1.0, 3
    h = (b - a) / n
    w = simpson_weights(a, b, n)
    assert np.isclose(w[0], 3 * h / 8, atol=1e-13), f"w[0]={w[0]}, expected 3h/8={3*h/8}"
    assert not np.isclose(w[0], 17 * h / 24, atol=1e-6), "x_0 must NOT be 17h/24 at n=3"
    expected = (3 * h / 8) * np.array([1.0, 3.0, 3.0, 1.0])
    assert np.allclose(w, expected, atol=1e-13), f"n=3 weights {w} != 3/8 rule {expected}"
    print('test_case_16 passed')



## Solution

In [ ]:
import numpy as np
from typing import Callable


def _validate_ab(a, b):
    if not (np.isfinite(a) and np.isfinite(b)):
        raise ValueError("a and b must be finite.")
    if a >= b:
        raise ValueError("a must be strictly less than b")
    return float(a), float(b)


def _validate_n(n):
    if isinstance(n, bool) or not isinstance(n, (int, np.integer)):
        raise TypeError(f"n must be an integer, got {type(n).__name__}.")
    n = int(n)
    if n < 2:
        raise ValueError(f"n must be >= 2, got n={n}.")
    return n


def _eval_scalar_finite(f, x):
    """Evaluate f(x) once, requiring a finite real scalar."""
    try:
        raw = f(x)
    except (OverflowError, ZeroDivisionError, FloatingPointError) as exc:
        raise ValueError(f"Integrand failed at x={x!r}: {exc}") from exc

    if isinstance(raw, np.ndarray):
        if raw.ndim != 0:
            raise TypeError(
                f"f(x) must return a scalar; at x={x!r} got array of shape {raw.shape}."
            )
        val = float(raw)
    elif np.isscalar(raw):
        val = float(raw)
    else:
        raise TypeError(
            f"f(x) must return a scalar; at x={x!r} got {type(raw).__name__}."
        )

    if not np.isfinite(val):
        raise ValueError(f"Integrand returned a non-finite value at x={x!r}: {val!r}.")
    return val


def _simpson_weight_vector(n, h):
    """Raw composite Simpson weights of length n+1, with panel width h=(b-a)/n.

    - Even n: (h/3) * [1, 4, 2, 4, ..., 4, 1].
    - Odd n: 1/3 rule on the first n-3 panels, 3/8 rule on the last 3.
      For odd n >= 5 the 1/3 head is non-empty and its right endpoint
      coincides with the 3/8 tail's left endpoint, so the shared
      transition node x_{n-3} receives the SUM of the two rules'
      contributions, h*(1/3 + 3/8) = 17h/24. For n = 3 the 1/3 head is
      empty (n-3 = 0 panels): the rule is the pure 3/8 rule with weights
      (3h/8)[1,3,3,1], so x_0 = x_{n-3} carries only 3h/8. This code
      special-cases that: the 17h/24 combined weight is used only when the
      1/3 head is non-empty (odd n >= 5).
    """
    w = np.zeros(n + 1, dtype=float)

    if n % 2 == 0:
        w[0] = 1.0
        w[n] = 1.0
        w[1:n:2] = 4.0
        w[2:n:2] = 2.0
        w *= h / 3.0
        return w

    # Odd n: head length m = n - 3 (even, >= 0).
    m = n - 3

    if m >= 2:
        # 1/3 rule on the first m panels, contributing to nodes 0..m.
        w[0]     += h / 3.0
        w[m]     += h / 3.0
        w[1:m:2] += 4.0 * h / 3.0
        w[2:m:2] += 2.0 * h / 3.0

    # 3/8 rule on the last 3 panels, contributing to nodes m..n.
    # At m=0 the head is empty and the combined weight h*(1/3 + 3/8) is
    # not needed: x_0 gets only the 3/8-rule endpoint weight 3h/8.
    w[m]     += 3.0 * h / 8.0
    w[m + 1] += 9.0 * h / 8.0
    w[m + 2] += 9.0 * h / 8.0
    w[n]     += 3.0 * h / 8.0

    return w


def simpson_weights(a: float, b: float, n: int) -> np.ndarray:
    """
    Raw integration weights of composite Simpson's rule on [a, b] with n
    equal subintervals.

    Returns a length-(n+1) vector w such that for the equally spaced nodes
    x_i = a + i*(b-a)/n,

        composite_simpson(f, a, b, n) == sum_i w[i] * f(x_i)

    for every integrand f. Each w[i] already includes the panel width
    h = (b-a)/n.

    For even n:  w = (h/3) * [1, 4, 2, 4, ..., 4, 1].
    For odd n:   1/3 rule on the first n-3 panels, 3/8 rule on the last 3.
                 For odd n >= 5 the shared transition node x_{n-3} carries
                 the combined weight h*(1/3 + 3/8) = 17h/24. For n = 3 the
                 1/3 head is empty, so the rule is the pure 3/8 rule and
                 x_0 = x_{n-3} carries only the 3/8 endpoint weight 3h/8.
    """
    a, b = _validate_ab(a, b)
    n = _validate_n(n)
    h = (b - a) / n
    return _simpson_weight_vector(n, h)


def composite_simpson(f: Callable[[float], float], a: float, b: float, n: int) -> float:
    """
    Composite Simpson quadrature on [a, b] with n equal subintervals,
    valid for any integer n >= 2 (even or odd).

    Even n: standard composite 1/3 rule on all n panels.
    Odd n:  composite 1/3 rule on the first n-3 panels, then Simpson's
            3/8 rule on the final three panels [x_{n-3}, x_n].

    Raises
    ------
    ValueError
        If a >= b, or if n < 2.
    """
    a, b = _validate_ab(a, b)
    n = _validate_n(n)

    h = (b - a) / n
    xs = np.linspace(a, b, n + 1)
    w = _simpson_weight_vector(n, h)

    vals = np.empty(n + 1, dtype=float)
    for i, x in enumerate(xs):
        vals[i] = _eval_scalar_finite(f, float(x))

    result = float(np.dot(w, vals))
    if not np.isfinite(result):
        raise FloatingPointError(f"Quadrature produced a non-finite result: {result!r}.")
    return result

# Subproblem 2

## Prompt

Implement a Richardson error estimate for `composite_simpson` by comparing the result with $n$ subintervals against the result with $2n$ subintervals.

> **Applicability of the even-power expansion (scientific caveat).** The
> Richardson machinery below assumes the composite Simpson error admits a
> pure *even-power* asymptotic expansion $I - S_n = C_4 h^4 + C_6 h^6 +
> \dots$ (only $h^4, h^6, h^8, \dots$). This holds for the pure composite
> 1/3 rule, i.e. for **even** $n$. It does **not** hold for the mixed
> 1/3 + 3/8 rule used at **odd** $n$: the 3/8 tail contributes an
> *odd-power* term, so the odd-$n$ error carries an $O(h^5)$ component.
> For example, for $f(x)=x^4$ on $[0,1]$ the mixed odd-$n$ error is exactly
> $\tfrac{2}{15\,n^4} + \tfrac{1}{2\,n^5}$, and the $r=2$ extrapolant then
> retains a residual $-\tfrac{1}{30\,n^5} = O(h^5)$ rather than the
> $O(h^6)$ one would get in the pure even-power case; the higher
> $2k+4$ exponent schedule likewise degrades once an odd-power term is
> present. Consequently the extrapolants and their $r^{4}-1$, $r^{2k+4}-1$
> error estimates below are exact-order only for **even** $n$; for odd $n$
> they remain valid *asymptotically* (the odd-power term is higher order
> than the leading $h^4$) and serve as order-of-magnitude error indicators,
> but should not be read as delivering the full $O(h^{2k+4})$ accuracy.


```python
def richardson_error_estimate(f: Callable, a: float, b: float, n: int,
                              r: int = 2) -> Tuple[float, float]:
    """
    Richardson error estimate comparing composite_simpson(f, a, b, n) with
    composite_simpson(f, a, b, r*n).

    Composite Simpson is fourth-order: refining by a factor r shrinks the
    leading truncation error by r**4, giving

        error = |S_rn - S_n| / (r**4 - 1),

    reducing to |S_2n - S_n| / 15 when r = 2.

    Returns (S_rn, error). Raises ValueError if a >= b, n < 2, or r < 2.
    """


def richardson_extrapolate(f: Callable, a: float, b: float, n: int,
                           r: int = 2) -> float:
    """
    First-order Richardson extrapolant at the r*n refinement level:

        S_extrap = (r**4 * S_rn - S_n) / (r**4 - 1),

    cancels the leading h**4 term. For even n (pure 1/3 rule, even-power
    expansion) this leaves an O(h**6) estimate; for odd n the mixed rule's
    O(h**5) term is only partly cancelled, leaving O(h**5).
    Raises ValueError if a >= b, n < 2, or r < 2.
    """


def richardson_iterated(f: Callable, a: float, b: float, n: int,
                        k: int, r: int = 2) -> float:
    """
    Apply k levels of Richardson extrapolation to composite Simpson. For
    even n (pure 1/3 rule) the error is a pure even-power series, so this
    cancels h**4, h**6, ..., h**(2k+2) and leaves an O(h**(2k+4)) estimate.
    For odd n the mixed 1/3+3/8 rule carries an odd-power O(h**5) term that
    this even-power schedule does not cancel, so the nominal 2j+2 exponents
    hold only asymptotically and the realized order is limited by that
    O(h**5) term.

    Define R_0^{(m)} = composite_simpson(f, a, b, m). The j-th iteration is

        R_j^{(m)} = (r**(2j+2) * R_{j-1}^{(r*m)} - R_{j-1}^{(m)})
                    / (r**(2j+2) - 1),

    and the required output is R_k^{(n)}. The exponent in the j-th
    iteration is 2j+2, NOT a constant 4: after j-1 iterations the leading
    surviving error term is h**(2j+2), and successive levels of that term
    scale as r**(2j+2). Concretely:

        k=1: R_1^{(n)} = (r**4 R_0^{(rn)} - R_0^{(n)}) / (r**4 - 1)
        k=2: R_2^{(n)} = (r**6 R_1^{(rn)} - R_1^{(n)}) / (r**6 - 1)
        k=3: R_3^{(n)} = (r**8 R_2^{(rn)} - R_2^{(n)}) / (r**8 - 1)

    Requires k+1 Simpson levels: n, rn, r**2 n, ..., r**k n.

    Raises ValueError if a >= b, n < 2, r < 2, or k < 1.
    """


def richardson_error_of_iterated(f: Callable, a: float, b: float, n: int,
                                 k: int, r: int = 2) -> Tuple[float, float]:
    """
    Error estimate OF the k-th level Richardson extrapolant R_k.

    After k iterations, R_k has cancelled h**4, ..., h**(2k+2), so its
    leading surviving error is C_{2k+4} h**(2k+4). To estimate that error,
    compare two adjacent R_k values, R_k^{(n)} and R_k^{(rn)}:

        error = |R_k^{(rn)} - R_k^{(n)}| / (r**(2k+4) - 1).

    Note the exponent is 2k+4, NOT 4. Reusing r**4 - 1 (or r**(2k+2) - 1)
    does not produce a valid error estimate of R_k.

    Parameters
    ----------
    f, a, b, n, r :
        Same conventions as richardson_error_estimate.
    k : int
        Number of Richardson iterations. Must be >= 1.

    Returns
    -------
    R_k^{(rn)} : float
        The k-th level extrapolant at the FINER (r*n) level -- the more
        accurate of the two extrapolants R_k^{(n)}, R_k^{(rn)}. (Returned
        for consistency with `richardson_error_estimate`, which likewise
        returns the finer value S_{rn} together with its own error.)
    error : float
        Non-negative estimate of that finer extrapolant's error, computed as
        |R_k^{(rn)} - R_k^{(n)}| / (r**(2k+4) - 1).

    Requires k+2 Simpson levels: n, rn, ..., r**(k+1) n.

    Raises ValueError if a >= b, n < 2, r < 2, or k < 1.
    """
    ```


## Background

Let $S_n$ be `composite_simpson(f, a, b, n)` and $S_{2n}$ be `composite_simpson(f, a, b, 2n)`. Since composite Simpson's rule is fourth-order accurate, halving the panel width (doubling $n$) shrinks the truncation error by a factor of $2^4 = 16$, which gives the standard Richardson extrapolation error estimate

$$\text{error} = \frac{|S_{2n} - S_n|}{15}.$$

Note that this requires calling `composite_simpson` with *two different* panel counts ($n$ and $2n$); since $n$ can now be odd, $S_n$ may use the 1/3+3/8 combination while $S_{2n}$ (always even) uses the pure 1/3 rule -- both are still fourth-order-accurate *asymptotically* (as $h \to 0$), so the ratio-of-16 argument for the error formula holds only in that asymptotic limit. Note moreover that the mixed odd-$n$ rule does **not** have a pure even-power error expansion: its 3/8 tail adds an $O(h^5)$ term (e.g. for $f=x^4$ on $[0,1]$ the odd-$n$ error is $\tfrac{2}{15n^4}+\tfrac{1}{2n^5}$), so the $r=2$ extrapolant leaves an $O(h^5)$ residual rather than $O(h^6)$; the even-power $2k+4$ schedule and the $O(h^6)/O(h^8)$ accuracy claims below are exact-order only for **even** $n$. **This estimate is the error of $S_{2n}$ (the finer of the pair), not of $S_n$**: since $S_{2n}=I+Ch^4/16+\dots$ and $S_n = I + Ch^4+\dots$, algebra gives $e_n = S_n - I \approx 16\,e_{2n}$ where $e_{2n}=S_{2n}-I$, so $S_{2n}-S_n = e_{2n}-e_n \approx -15\,e_{2n} = -15(S_{2n}-I)$, giving $|S_{2n}-I|\approx|S_{2n}-S_n|/15$. At small $n$ -- including the $n=2$ leaf level used by the Main Problem -- the two constants $C$ implicit in $S_n$ (mixed rule) and $S_{2n}$ (pure rule) need not match closely, and the observed error ratio can deviate substantially from 16 (it can even be less than 1 for some $n$); the formula should be read as an order-of-magnitude error indicator at low panel counts, not a tight guarantee, converging to a tight estimate only as $n$ grows. The error estimate is only meaningful when $S_{2n}$ is obtained by doubling every subinterval of the $S_n$ mesh, not by an arbitrary unrelated refinement.
**General refinement ratio.** The `r` parameter generalizes the r=2 case.
Because composite Simpson's rule is fourth-order accurate, refining the mesh
by a factor r reduces the truncation error by r**4, giving

    error = |S_rn - S_n| / (r**4 - 1).

This requires that S_rn is obtained by subdividing each panel of the S_n
mesh into r equal pieces; it is not valid for arbitrary unrelated
refinements. The r=2 case is the classic doubling comparison, and any
implementation that hardcodes /15 or 2*n regardless of r fails the general
formula.

**Connection to adaptive quadrature.** The Main Problem instantiates this general
argument at the fixed special case r=2, n=2 at every recursive leaf: the pair
(S_n, S_rn) = (S_2, S_4) is used both to form the error estimate
|S_4 - S_2| / (2**4 - 1) = |S_4 - S_2| / 15, and — because S_4 is already the
more accurate of the two values computed — to supply the accepted leaf value
itself when that error is within tolerance (local extrapolation; see the Main
Problem Background). The general r**4 - 1 denominator derived here is exactly
what must be reused, unmodified, inside the adaptive recursion rather than
being reimplemented or hardcoded as `/15`.


## Testing Template

In [ ]:
# ============ TEST CASES ============
import numpy as np
import math


def test_case_1():
    f = lambda x: np.sin(x) + 0.3 * x**2
    a, b, n, r = 0.5, 2.3, 4, 3
    S_rn, err = richardson_error_estimate(f, a, b, n, r=r)
    expected_S_rn = composite_simpson(f, a, b, r * n)
    S_n = composite_simpson(f, a, b, n)
    assert abs(S_rn - expected_S_rn) < 1e-12, (
        f"first element must be S_rn = composite_simpson(f,a,b,{r*n}), "
        f"got {S_rn}, expected {expected_S_rn}"
    )
    assert abs(S_rn - S_n) > 1e-6
    print('test_case_1 passed')


def test_case_2():
    f = lambda x: x**6
    a, b, n, r = 0.0, 1.0, 4, 3
    S_ex = richardson_extrapolate(f, a, b, n, r=r)
    S_n = composite_simpson(f, a, b, n)
    S_rn = composite_simpson(f, a, b, r * n)
    correct = (r**4 * S_rn - S_n) / (r**4 - 1)
    wrong_hardcoded = (16 * S_rn - S_n) / 15
    assert abs(S_ex - correct) < 1e-12
    assert abs(S_ex - wrong_hardcoded) > 1e-4
    print('test_case_2 passed')


def test_case_3():
    R2 = richardson_iterated(lambda x: x**6, 0.0, 1.0, 2, 2, r=3)
    assert abs(R2 - 1.0/7.0) < 1e-10, f"got {R2}, expected 1/7"
    print('test_case_3 passed')


def test_case_4():
    f = lambda x: np.exp(x) * np.sin(2 * x)
    a, b = 0.2, 1.7
    for n, r in [(2, 2), (4, 3), (6, 2), (3, 5)]:
        R1 = richardson_iterated(f, a, b, n, 1, r=r)
        E1 = richardson_extrapolate(f, a, b, n, r=r)
        assert abs(R1 - E1) < 1e-12, (
            f"k=1 iterated must equal extrapolate at n={n}, r={r}"
        )
    print('test_case_4 passed')

def test_case_6():
    f = lambda x: x**4
    a, b, n = 0.0, 1.0, 2
    S_rn, err = richardson_error_estimate(f, a, b, n)
    assert abs(S_rn - 77.0/384.0) < 1e-12
    assert abs(err - 1.0/1920.0) < 1e-12
    S_ex = richardson_extrapolate(f, a, b, n)
    assert abs(S_ex - 0.2) < 1e-12
    print('test_case_6 passed')


def test_case_7():
    f = lambda x: x**5
    a, b, n = 0.0, 1.0, 5
    S_rn, err = richardson_error_estimate(f, a, b, n)
    assert abs(S_rn - 1667.0/10000.0) < 1e-12
    assert abs(err - 53.0/750000.0) < 1e-12
    print('test_case_7 passed')


def test_case_8():
    f = lambda x: np.exp(x)
    F_exact = math.exp(1.0) - 1.0
    for k in [4, 6, 8]:
        R_k = richardson_iterated(f, 0.0, 1.0, 2, k, r=2)
        assert np.isfinite(R_k), f"k={k}: non-finite result {R_k}"
        assert abs(R_k - F_exact) < 1e-10, (
            f"k={k}: R_k = {R_k}, exact = {F_exact}"
        )
    print('test_case_8 passed')


def test_case_9():
    for k in [1, 2, 3, 4]:
        deg = 2 * k + 2
        f = lambda x, d=deg: x ** d
        R_k = richardson_iterated(f, 0.0, 1.0, 2, k, r=2)
        expected = 1.0 / (deg + 1)
        assert abs(R_k - expected) < 1e-9, (
            f"k={k}: R_k = {R_k}, expected {expected}"
        )
    print('test_case_9 passed')


def test_case_10():
    f = lambda x: np.cos(2 * x) + 0.5 * x
    a, b, n, r = 0.3, 2.1, 4, 2
    R2 = richardson_iterated(f, a, b, n, 2, r=r)

    def R1_at(m):
        Sn_ = composite_simpson(f, a, b, m)
        Srn_ = composite_simpson(f, a, b, r * m)
        return (r**4 * Srn_ - Sn_) / (r**4 - 1)

    R1_n = R1_at(n)
    R1_rn = R1_at(r * n)
    factor = r ** 6
    expected_R2 = (factor * R1_rn - R1_n) / (factor - 1)
    assert abs(R2 - expected_R2) < 1e-12
    print('test_case_10 passed')


def test_case_11():
    f = lambda x: x**8
    _, e_small = richardson_error_of_iterated(f, 0.0, 1.0, 2, 2)
    _, e_large = richardson_error_of_iterated(f, 0.0, 1.0, 4, 2)
    ratio = e_small / e_large
    assert 0.3 * 256 < ratio < 3.0 * 256, (
        f"expected ratio ~256 (r**8), got {ratio:.2f}"
    )
    print('test_case_11 passed')


def test_case_12():
    f = lambda x: np.exp(x)
    prev_err = None
    for k in [1, 2, 3, 4, 5]:
        _, err = richardson_error_of_iterated(f, 0.0, 1.0, 2, k)
        assert np.isfinite(err)
        if prev_err is not None:
            assert err < prev_err, (
                f"error should shrink with k; at k={k}, "
                f"err={err} >= prev={prev_err}"
            )
        prev_err = err
    print('test_case_12 passed')


def test_case_13():
    for fn in [richardson_error_estimate, richardson_extrapolate,
               richardson_iterated, richardson_error_of_iterated]:
        try:
            if fn is richardson_iterated or fn is richardson_error_of_iterated:
                fn(lambda x: x, 1.0, 0.0, 3, 1)
            else:
                fn(lambda x: x, 1.0, 0.0, 3)
            assert False, f"{fn.__name__} did not raise for a >= b"
        except ValueError:
            pass
        try:
            if fn is richardson_iterated or fn is richardson_error_of_iterated:
                fn(lambda x: x, 0.0, 1.0, 1, 1)
            else:
                fn(lambda x: x, 0.0, 1.0, 1)
            assert False, f"{fn.__name__} did not raise for n < 2"
        except ValueError:
            pass
    print('test_case_13 passed')


def test_case_14():
    for k_bad in [0, -1]:
        try:
            richardson_iterated(lambda x: x, 0.0, 1.0, 3, k_bad)
            assert False
        except (ValueError, TypeError):
            pass
        try:
            richardson_error_of_iterated(lambda x: x, 0.0, 1.0, 3, k_bad)
            assert False
        except (ValueError, TypeError):
            pass
    for r_bad in [1, 0, -1]:
        try:
            richardson_error_estimate(lambda x: x, 0.0, 1.0, 3, r=r_bad)
            assert False
        except (ValueError, TypeError):
            pass
    print('test_case_14 passed')

## Solution

In [ ]:
import numpy as np
from typing import Callable, Tuple

_INT_MAX = np.iinfo(np.int64).max


# ---------------------------------------------------------------------------
# Shared validation and integrand evaluation
# ---------------------------------------------------------------------------

def _validate_ab(a, b) -> Tuple[float, float]:
    if not (np.isfinite(a) and np.isfinite(b)):
        raise ValueError("a and b must be finite.")
    if a >= b:
        raise ValueError("a must be strictly less than b")
    return float(a), float(b)


def _validate_n(n) -> int:
    if isinstance(n, bool) or not isinstance(n, (int, np.integer)):
        raise TypeError(f"n must be an integer, got {type(n).__name__}.")
    n = int(n)
    if n < 2:
        raise ValueError(f"n must be >= 2, got n={n}.")
    return n


def _validate_r(r) -> int:
    if isinstance(r, bool) or not isinstance(r, (int, np.integer)):
        raise TypeError(f"r must be an integer, got {type(r).__name__}.")
    r = int(r)
    if r < 2:
        raise ValueError(f"r must be >= 2, got r={r}.")
    return r


def _validate_k(k) -> int:
    if isinstance(k, bool) or not isinstance(k, (int, np.integer)):
        raise TypeError(f"k must be an integer, got {type(k).__name__}.")
    k = int(k)
    if k < 1:
        raise ValueError(f"k must be >= 1, got k={k}.")
    return k


def _eval_scalar_finite(f, x: float) -> float:
    try:
        raw = f(x)
    except (OverflowError, ZeroDivisionError, FloatingPointError) as exc:
        raise ValueError(f"Integrand failed at x={x!r}: {exc}") from exc
    if isinstance(raw, np.ndarray):
        if raw.ndim != 0:
            raise TypeError(f"f(x) must return a scalar; got shape {raw.shape}.")
        val = float(raw)
    elif np.isscalar(raw):
        val = float(raw)
    else:
        raise TypeError(f"f(x) must return a scalar; got {type(raw).__name__}.")
    if not np.isfinite(val):
        raise ValueError(f"Non-finite integrand value at x={x!r}: {val!r}.")
    return val


# ---------------------------------------------------------------------------
# Composite Simpson (inlined from Subproblem 1)
# ---------------------------------------------------------------------------

def _simpson_weight_vector(n: int, h: float) -> np.ndarray:
    w = np.zeros(n + 1, dtype=float)
    if n % 2 == 0:
        w[0] = w[n] = 1.0
        w[1:n:2] = 4.0
        w[2:n:2] = 2.0
        w *= h / 3.0
        return w
    m = n - 3
    if m >= 2:
        w[0]     += h / 3.0
        w[m]     += h / 3.0
        w[1:m:2] += 4.0 * h / 3.0
        w[2:m:2] += 2.0 * h / 3.0
    w[m]     += 3.0 * h / 8.0
    w[m + 1] += 9.0 * h / 8.0
    w[m + 2] += 9.0 * h / 8.0
    w[n]     += 3.0 * h / 8.0
    return w


def composite_simpson(f: Callable[[float], float], a: float, b: float,
                      n: int) -> float:
    a, b = _validate_ab(a, b)
    n = _validate_n(n)
    h = (b - a) / n
    xs = np.linspace(a, b, n + 1)
    w = _simpson_weight_vector(n, h)
    vals = np.empty(n + 1, dtype=float)
    for i, x in enumerate(xs):
        vals[i] = _eval_scalar_finite(f, float(x))
    result = float(np.dot(w, vals))
    if not np.isfinite(result):
        raise FloatingPointError(f"Quadrature produced {result!r}.")
    return result


# ---------------------------------------------------------------------------
# Richardson: single-step error estimate and first-order extrapolant
# ---------------------------------------------------------------------------

def richardson_error_estimate(f: Callable, a: float, b: float, n: int,
                              r: int = 2) -> Tuple[float, float]:
    """
    Richardson error estimate comparing composite_simpson(f, a, b, n) with
    composite_simpson(f, a, b, r*n).

    With S_n = S_n and S_rn the two estimates, and composite Simpson's
    fourth-order accuracy, the standard estimate is

        error = |S_rn - S_n| / (r**4 - 1),

    reducing to |S_2n - S_n| / 15 for r = 2. This estimate is the error of
    the FINER of the pair, S_rn (see Background); consistent with that, this
    function returns S_rn as its first element.
    """
    a, b = _validate_ab(a, b)
    n = _validate_n(n)
    r = _validate_r(r)
    if n > _INT_MAX // r:
        raise ValueError(f"n too large: r*n would overflow, got n={n}, r={r}.")
    S_n  = composite_simpson(f, a, b, n)
    S_rn = composite_simpson(f, a, b, r * n)
    err = abs(S_rn - S_n) / (r ** 4 - 1)
    if not np.isfinite(err):
        raise FloatingPointError(f"Richardson estimate is {err!r}.")
    return S_rn, float(err)


def richardson_extrapolate(f: Callable, a: float, b: float, n: int,
                           r: int = 2) -> float:
    """
    First-order Richardson extrapolant at the r*n refinement level:

        S_extrap = (r**4 * S_rn - S_n) / (r**4 - 1),

    cancelling the leading h**4 term and leaving an O(h**6) estimate.
    """
    a, b = _validate_ab(a, b)
    n = _validate_n(n)
    r = _validate_r(r)
    if n > _INT_MAX // r:
        raise ValueError(f"n too large: r*n would overflow, got n={n}, r={r}.")
    S_n  = composite_simpson(f, a, b, n)
    S_rn = composite_simpson(f, a, b, r * n)
    r4 = r ** 4
    val = (r4 * S_rn - S_n) / (r4 - 1)
    if not np.isfinite(val):
        raise FloatingPointError(f"Richardson extrapolant is {val!r}.")
    return float(val)


# ---------------------------------------------------------------------------
# Iterated Richardson (k levels) and error estimate
# ---------------------------------------------------------------------------

def _build_richardson_column(S, r, k):
    """Column-wise Richardson iteration.

    Input: list S of Simpson values at levels n, rn, ..., r^k n (length k+1).
    Returns: the final column value R_k^{(n)} after k iterations. The j-th
    iteration uses exponent 2*j+2: for j=1, 4; for j=2, 6; etc.
    """
    S = list(S)
    for j in range(1, k + 1):
        exponent = 2 * j + 2
        factor = r ** exponent
        S = [(factor * S[i + 1] - S[i]) / (factor - 1)
             for i in range(len(S) - 1)]
    return S[0]


def richardson_iterated(f: Callable, a: float, b: float, n: int,
                        k: int, r: int = 2) -> float:
    """
    Apply k levels of Richardson extrapolation to composite Simpson.
    The j-th iteration uses exponent 2*j+2 (i.e. 4, 6, 8, ... for
    j = 1, 2, 3, ...), cancelling h**(2*j+2) each time.
    """
    a, b = _validate_ab(a, b)
    n = _validate_n(n)
    r = _validate_r(r)
    k = _validate_k(k)
    if n * (r ** k) > _INT_MAX:
        raise ValueError(f"n * r**k would overflow, got n={n}, r={r}, k={k}.")

    levels = [n * (r ** i) for i in range(k + 1)]
    S = [composite_simpson(f, a, b, m) for m in levels]
    result = float(_build_richardson_column(S, r, k))
    if not np.isfinite(result):
        raise FloatingPointError(f"Iterated Richardson value is {result!r}.")
    return result


def richardson_error_of_iterated(f: Callable, a: float, b: float, n: int,
                                 k: int, r: int = 2) -> Tuple[float, float]:
    """
    Error estimate of the k-th level Richardson extrapolant, returned
    together with the *finer* of the two adjacent extrapolant values from
    which that error estimate is derived.

    After k iterations, R_k has cancelled h**4, ..., h**(2k+2), so its
    leading surviving error is C_{2k+4} h**(2k+4). Writing h = h_n and
    using h_{rn} = h_n / r,

        R_k^{(n)}  - I  ~  C h**(2k+4)
        R_k^{(rn)} - I  ~  C (h/r)**(2k+4)
                        =  (1 / r**(2k+4)) * (R_k^{(n)} - I)

    Subtracting gives

        R_k^{(rn)} - R_k^{(n)}  ~  -C h**(2k+4) (r**(2k+4) - 1) / r**(2k+4)

    so

        |R_k^{(rn)} - I|  =  |R_k^{(rn)} - R_k^{(n)}| / (r**(2k+4) - 1)
        |R_k^{(n)}  - I|  =  r**(2k+4) * |R_k^{(rn)} - R_k^{(n)}|
                                           / (r**(2k+4) - 1)

    i.e. the standard Richardson estimate |R_k^{(rn)} - R_k^{(n)}| /
    (r**(2k+4) - 1) is the error of the FINER value R_k^{(rn)}, NOT of
    the coarser R_k^{(n)}. (The coarser value's error is r**(2k+4) times
    this expression.)

    Consistently with richardson_error_estimate -- which returns the
    finer value S_rn together with the error of S_rn -- this function
    returns (R_k^{(rn)}, err_of_R_k^{(rn)}).

    Note the exponent is 2k+4, NOT 4.

    Parameters
    ----------
    f, a, b, n, r :
        Same conventions as richardson_error_estimate.
    k : int
        Number of Richardson iterations. Must be >= 1.

    Returns
    -------
    R_k^{(rn)} : float
        The k-th level extrapolant at the finer (r*n) level. This is the
        more accurate of the two values R_k^{(n)}, R_k^{(rn)}.
    error : float
        Non-negative estimate of R_k^{(rn)}'s error, computed as
        |R_k^{(rn)} - R_k^{(n)}| / (r**(2k+4) - 1).

    Requires k+2 Simpson levels: n, rn, ..., r**(k+1) n.

    Raises ValueError if a >= b, n < 2, r < 2, or k < 1.
    """
    a, b = _validate_ab(a, b)
    n = _validate_n(n)
    r = _validate_r(r)
    k = _validate_k(k)
    if n * (r ** (k + 1)) > _INT_MAX:
        raise ValueError(
            f"n * r**(k+1) would overflow, got n={n}, r={r}, k={k}."
        )

    # k+2 levels: n, rn, r^2 n, ..., r^(k+1) n
    levels = [n * (r ** i) for i in range(k + 2)]
    Simpson = [composite_simpson(f, a, b, m) for m in levels]

    # Build the full Richardson table once. After k iterations S has
    # length k+2 - k = 2: S[0] = R_k^{(n)}, S[1] = R_k^{(rn)}.
    S = list(Simpson)
    for j in range(1, k + 1):
        exponent = 2 * j + 2
        factor = r ** exponent
        S = [(factor * S[i + 1] - S[i]) / (factor - 1)
             for i in range(len(S) - 1)]

    R_k_n  = S[0]
    R_k_rn = S[1]

    exponent = 2 * k + 4
    factor = r ** exponent
    err = abs(R_k_rn - R_k_n) / (factor - 1)
    if not np.isfinite(err):
        raise FloatingPointError(f"Extrapolant error estimate is {err!r}.")

    # Return the FINER extrapolant, paired with the error of that finer
    # value -- consistent with richardson_error_estimate, which returns
    # the finer S_rn together with |S_rn - S_n| / (r**4 - 1).
    return float(R_k_rn), float(err)

# Main Problem

## Prompt

Using `composite_simpson` and `richardson_error_estimate` (with the coarse panel count fixed at n=2 at each recursive leaf, matching the original single-application Simpson comparison), implement adaptive Simpson quadrature: recursively subdivide the integration interval until the estimated error on every piece is within tolerance.

```python
def main_problem(f: Callable, a: float, b: float, tol: float = 1e-8,
                  max_depth: int = 50) -> float:
    \"\"\"
    Parameters
    ----------
    f : callable
        f(x) -> scalar.
    a, b : float
        Integration interval, a < b.
    tol : float
        Target total error.
    max_depth : int
        Maximum recursion depth per branch. Must be >= 0. A value of 0
        performs no bisection and returns the root-level S4 estimate on
        [a, b]; only negative values are invalid.

    Raises
    ------
    ValueError
        If a >= b, if tol <= 0, or if max_depth < 0. (max_depth == 0 is valid: it means no recursion — the root-level S4 estimate on [a, b] is returned.)

    Returns
    -------
    integral : float
        Approximation of the integral of f over [a, b].
    \"\"\"
    ...
    return integral
```


## Background

Adaptive Simpson quadrature recursively applies `richardson_error_estimate(f, a_sub, b_sub, 2)` on each interval (i.e. comparing the n=2 and n=4 composite Simpson estimates for that interval): if the estimated error is within the current tolerance (or the recursion depth limit is reached), the **finer, four-panel estimate $S_4$** (returned as the first element, `S_rn`, of `richardson_error_estimate`) is accepted for that interval rather than the coarser two-panel estimate $S_2$. Accepting the finer of the two computed estimates is the standard choice: $S_4$ uses twice as many panels and, in the asymptotic (well-resolved) regime that the accepted-with-tolerance branch targets, is the more accurate of the pair. This is *not* a guarantee at every panel count -- as the Subproblem 2 Background notes, at very low panel counts the error ratio can fall below 1, so on a badly under-resolved subinterval $S_4$ need not beat $S_2$; but the acceptance test only fires once the Richardson error estimate is within tolerance (or the depth cap is hit), i.e. precisely where the asymptotic ordering is expected to hold. Note that this is plain selection of the finer Simpson value, **not** Richardson local extrapolation: the extrapolant would be $(16 S_4 - S_2)/15$, which is not what is accepted here. Otherwise the interval is bisected at its midpoint and the same procedure is applied to each half, with the tolerance halved so that the sum of the two sub-tolerances still equals the original tolerance for the parent interval. Summing the accepted estimates from every leaf of this recursion gives the overall integral estimate, with computational effort automatically concentrated on subintervals where $f$ is hardest to approximate (e.g. near sharp peaks or high curvature) while flatter regions are resolved with very few evaluations.


## Testing Template

In [ ]:
# ============ TEST CASES ============

import math
import numpy as np


def test_case_1():
    """Smooth Gaussian: adaptive result matches the known analytic value."""
    f = lambda x: np.exp(-x**2)
    result = main_problem(f, -1, 1, tol=1e-6, max_depth=20)
    expected = 1.4936482656248540508
    assert np.allclose(result, expected, rtol=3e-7, atol=3e-7)
    print('test_case_1 passed')
    return True

def test_case_2():
    """Cubic polynomial: Simpson is exact, so the leaf estimate is accepted immediately."""
    f = lambda x: 3*x**3 - x + 2
    a, b = -1.0, 2.0
    F = lambda x: 0.75*x**4 - 0.5*x**2 + 2*x
    expected = F(b) - F(a)
    result = main_problem(f, a, b, tol=1e-9, max_depth=10)
    assert np.allclose(result, expected, atol=1e-11)
    print('test_case_2 passed')
    return True

def test_case_3():
    """Highly oscillatory function integrates to (near) zero over a full period-ish span."""
    f = lambda x: np.sin(100 * x)
    result = main_problem(f, 0, np.pi, tol=1e-4, max_depth=35)
    expected = (1 - np.cos(100 * np.pi)) / 100.0
    assert abs(result - expected) < 1e-6
    print('test_case_3 passed')
    return True

def test_case_4():
    """Integrable singularity near the left endpoint: result stays close to the
    analytic antiderivative despite unbounded local curvature."""
    f = lambda x: 1.0 / np.sqrt(x)
    a = 1e-6
    result = main_problem(f, a, 1.0, tol=1e-3, max_depth=30)
    expected = 2.0 - 2.0 * np.sqrt(a)
    assert np.allclose(result, expected, atol=5e-4)
    print('test_case_4 passed')
    return True

def test_case_5():
    """Input validation matches the documented Raises contract."""
    try:
        main_problem(lambda x: x, 1.0, 0.0, tol=1e-6)
        assert False, 'expected ValueError for a >= b'
    except ValueError:
        pass
    try:
        main_problem(lambda x: x, 0.0, 1.0, tol=-1e-6)
        assert False, 'expected ValueError for tol <= 0'
    except ValueError:
        pass
    # max_depth == 0 is VALID: no recursion, returns the root-level S4 estimate.
    val0 = main_problem(lambda x: x, 0.0, 1.0, tol=1e-6, max_depth=0)
    assert np.isfinite(val0), 'max_depth=0 should return a finite root-level estimate'
    # only NEGATIVE max_depth is invalid.
    try:
        main_problem(lambda x: x, 0.0, 1.0, tol=1e-6, max_depth=-1)
        assert False, 'expected ValueError for max_depth < 0'
    except ValueError:
        pass
    print('test_case_5 passed')
    return True

def test_case_6():
    """HARDER: the requested tol must genuinely control the achieved accuracy,
    not just be accepted and ignored. A loose and a tight tol on the same
    smooth function must each meet their own tolerance, and the tight run
    must be dramatically more accurate."""
    f = lambda x: np.exp(-x) * np.sin(3*x)
    F = lambda x: np.exp(-x) * (-np.sin(3*x) - 3*np.cos(3*x)) / 10.0
    a, b = 0.0, 2.0
    exact = F(b) - F(a)

    loose = main_problem(f, a, b, tol=1e-3, max_depth=30)
    tight = main_problem(f, a, b, tol=1e-9, max_depth=30)
    err_loose = abs(loose - exact)
    err_tight = abs(tight - exact)

    assert err_loose < 1e-3, f"loose-tol run should meet its own tol, got err={err_loose:.3e}"
    assert err_tight < 1e-9, f"tight-tol run should meet its own tol, got err={err_tight:.3e}"
    assert err_tight < err_loose / 100, "tightening tol should sharply improve accuracy"
    print('test_case_6 passed')
    return True

def test_case_7():
    """HARDER: max_depth must be a genuine, enforced cap on recursion, not a
    parameter that is accepted and then ignored. A narrow Gaussian bump needs
    real recursive refinement to resolve: starving the recursion of depth
    must leave a measurably worse (and tol-violating) result, while enough
    depth must resolve it comfortably within tol."""
    k, c, a, b = 200.0, 0.37, 0.0, 1.0
    f = lambda x: np.exp(-k * (x - c)**2)
    exact = math.sqrt(math.pi / k) / 2.0 * (
        math.erf(math.sqrt(k) * (b - c)) - math.erf(math.sqrt(k) * (a - c))
    )
    shallow = main_problem(f, a, b, tol=1e-6, max_depth=2)
    deep = main_problem(f, a, b, tol=1e-6, max_depth=20)
    err_shallow = abs(shallow - exact)
    err_deep = abs(deep - exact)

    assert err_deep < 1e-5, f"depth-sufficient run should meet tol, got err={err_deep:.3e}"
    assert err_shallow > 1e-4, (
        f"depth-starved run should NOT meet tol if max_depth is truly enforced, "
        f"got err={err_shallow:.3e}"
    )
    assert err_deep < err_shallow / 100, "deeper recursion must be substantially more accurate"
    print('test_case_7 passed')
    return True




## Solution

In [ ]:
import math
import numpy as np
from typing import Callable, Tuple

_INT_MAX = np.iinfo(np.int64).max


# ---------------------------------------------------------------------------
# Shared validation
# ---------------------------------------------------------------------------

def _validate_ab(a, b) -> Tuple[float, float]:
    if not (np.isfinite(a) and np.isfinite(b)):
        raise ValueError("a and b must be finite.")
    if a >= b:
        raise ValueError(f"Require a < b, got a={a}, b={b}.")
    return float(a), float(b)


def _validate_n(n, *, minimum: int = 2) -> int:
    if isinstance(n, bool) or not isinstance(n, (int, np.integer)):
        raise TypeError(f"n must be an integer, got {type(n).__name__}.")
    n = int(n)
    if n < minimum:
        raise ValueError(f"n must be >= {minimum}, got {n}.")
    return n


def _validate_r(r) -> int:
    if isinstance(r, bool) or not isinstance(r, (int, np.integer)):
        raise TypeError(f"r must be an integer, got {type(r).__name__}.")
    r = int(r)
    if r < 2:
        raise ValueError(f"r must be >= 2, got r={r}.")
    return r


def _eval_scalar_finite(f, x: float) -> float:
    try:
        raw = f(x)
    except (OverflowError, ZeroDivisionError, FloatingPointError) as exc:
        raise ValueError(f"Integrand failed at x={x!r}: {exc}") from exc
    if isinstance(raw, np.ndarray):
        if raw.ndim != 0:
            raise TypeError(f"f(x) must return a scalar; got shape {raw.shape}.")
        val = float(raw)
    elif np.isscalar(raw):
        val = float(raw)
    else:
        raise TypeError(f"f(x) must return a scalar; got {type(raw).__name__}.")
    if not np.isfinite(val):
        raise ValueError(f"Non-finite integrand value at x={x!r}: {val!r}.")
    return val


# ---------------------------------------------------------------------------
# Composite Simpson (inlined from Subproblem 1)
# ---------------------------------------------------------------------------

def _simpson_weights(n: int, h: float) -> np.ndarray:
    w = np.zeros(n + 1, dtype=float)
    if n % 2 == 0:
        w[0] = w[n] = 1.0
        w[1:n:2] = 4.0
        w[2:n:2] = 2.0
        w *= h / 3.0
        return w
    m = n - 3
    if m >= 2:
        w[0]     += h / 3.0
        w[m]     += h / 3.0
        w[1:m:2] += 4.0 * h / 3.0
        w[2:m:2] += 2.0 * h / 3.0
    w[m]     += 3.0 * h / 8.0
    w[m + 1] += 9.0 * h / 8.0
    w[m + 2] += 9.0 * h / 8.0
    w[n]     += 3.0 * h / 8.0
    return w


def composite_simpson(f, a, b, n) -> float:
    a, b = _validate_ab(a, b)
    n = _validate_n(n)
    h = (b - a) / n
    xs = np.linspace(a, b, n + 1)              # anchored endpoints
    w = _simpson_weights(n, h)
    vals = np.array([_eval_scalar_finite(f, float(x)) for x in xs])
    result = math.fsum(w[i] * vals[i] for i in range(n + 1))
    if not np.isfinite(result):
        raise FloatingPointError(f"Quadrature produced {result!r}.")
    return result


# ---------------------------------------------------------------------------
# Richardson estimate (inlined from Subproblem 2) -- the GENERAL,
# r-parameterized version, not a narrower n=2/r=2-only special case: it
# takes the same (f, a, b, n, r=2) signature and uses the general
# error = |S_rn - S_n| / (r**4 - 1) formula, exactly as Subproblem 2
# defines it. main_problem below calls it at its n=2 special case (r
# left at its default of 2) rather than this function being rewritten
# to hardcode 2*n and /15.
# ---------------------------------------------------------------------------

def richardson_error_estimate(f, a, b, n, r: int = 2) -> Tuple[float, float]:
    """error = |S_rn - S_n| / (r**4 - 1); returns (S_rn, error)."""
    a, b = _validate_ab(a, b)
    n = _validate_n(n)
    r = _validate_r(r)
    if n > _INT_MAX // r:
        raise ValueError(f"n too large: r*n would overflow, got n={n}, r={r}.")
    S_n  = composite_simpson(f, a, b, n)
    S_rn = composite_simpson(f, a, b, r * n)
    err = abs(S_rn - S_n) / (r ** 4 - 1)
    if not np.isfinite(err):
        raise FloatingPointError(f"Richardson estimate is {err!r}.")
    return S_rn, float(err)

# ---------------------------------------------------------------------------
# Adaptive Simpson (recursive, tolerance-halving per the spec)
# ---------------------------------------------------------------------------

def main_problem(f, a, b, tol: float = 1e-8, max_depth: int = 50) -> float:
    """
    Adaptive Simpson quadrature.

    Recursively applies richardson_error_estimate(f, a_sub, b_sub, 2) (the
    general Subproblem 2 helper, at its n=2, default r=2 special case) on
    each interval, comparing the n=2 and n=4 composite Simpson estimates.
    If the estimated error is within the current tolerance, or the
    recursion depth limit is reached, the finer n=4 estimate -- S4, i.e.
    the `S_rn` returned by richardson_error_estimate -- is accepted for
    that interval rather than the coarser S2. This is plain selection of
    the finer Simpson value (not Richardson extrapolation): S4 uses twice
    as many panels and is the more accurate of the two in the asymptotic
    regime the tolerance test targets, though not guaranteed to beat S2 at
    every panel count. Otherwise the interval is bisected at its midpoint and the same
    procedure is applied to each half with the tolerance halved, so that
    the sum of the two sub-tolerances equals the parent interval's
    tolerance.
    """
    a, b = _validate_ab(a, b)
    if not np.isfinite(tol) or tol <= 0:
        raise ValueError("tol must be positive and finite.")
    if isinstance(max_depth, bool) or not isinstance(max_depth, (int, np.integer)):
        raise TypeError(f"max_depth must be an integer, got {type(max_depth).__name__}.")
    if max_depth < 0:
        raise ValueError(f"max_depth must be non-negative, got {max_depth}.")

    def _recurse(a_sub, b_sub, tol_sub, depth):
        S4, err = richardson_error_estimate(f, a_sub, b_sub, 2)
        if err <= tol_sub or depth <= 0:
            return S4
        c = 0.5 * (a_sub + b_sub)
        if c == a_sub or c == b_sub:
            # Interval too small to split further in floating point.
            return S4
        half_tol = tol_sub / 2.0
        left = _recurse(a_sub, c, half_tol, depth - 1)
        right = _recurse(c, b_sub, half_tol, depth - 1)
        return left + right

    result = _recurse(a, b, tol, max_depth)
    if not np.isfinite(result):
        raise FloatingPointError(f"Adaptive Simpson produced a non-finite result: {result!r}.")
    return float(result)
